In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_11.pickle

In [4]:
%%RecordEvent
%%time
### cell 11 ###

# Step 1: read all essays into Python lists (list comprehensions are slightly more efficient than manual loop)
ids = [p.rsplit('/', 1)[-1].replace('.txt', '') for p in tqdm(train_txt)]
texts = [open(p, 'r').read() for p in tqdm(train_txt)]

# Step 2: build a GPU-backed DataFrame via pandas shim
# (the %load_ext cudf.pandas extension makes pandas.DataFrame GPU-backed)
df_text = pd.DataFrame({'id': ids, 'text': texts})

# Step 3: vectorized string ops + cast to int64
df_text['essay_len'] = (
    df_text['text']
      .str.strip()
      .str.len()
      .astype('int64')
)
# use regex count of contiguous non-whitespace as word count instead of split->list for better performance
df_text['essay_words'] = (
    df_text['text']
      .str.count(r"\S+")
      .astype('int64')
)

# Step 4: map stats back into `train` without a full merge
df_stats = df_text.set_index('id')[['essay_len', 'essay_words']]
train['essay_len']   = train['id'].map(df_stats['essay_len'])
train['essay_words'] = train['id'].map(df_stats['essay_words'])

  0%|          | 0/15594 [00:00<?, ?it/s]

  0%|          | 0/15594 [00:00<?, ?it/s]

CPU times: user 548 ms, sys: 245 ms, total: 794 ms
Wall time: 786 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/o4_mini_high_small/checkpoints/post_cell_11_try_7.pickle